<a href="https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

**Finding #4 — The Freshness Multiplier** (365+ day refreshed content
shows a 3.2x health boost and 57x more impressions).

Where the label comes from: health score and impression counts
measured before/after a refresh, bucketed by "days since last
update." Does the validation design carry the claim? Not fully —
the paper itself admits the 361+ freshness bucket is unstable (a
283:1 growth ratio built on just 1 declining page), and the "local
active-content" subset only includes pages with existing impressions
and sessions, which introduces survivor bias: pages that fully died
and dropped out of the active sample are invisible to this
comparison. So the 3.2x/57x headline is likely real directionally,
but probably overstated by which pages survived into the sample.

**ML Appendix — Feature Importance for Health Score** (Random Forest:
Average Position 43% importance, Impressions 32%, Scroll Depth 15%).

Where the label comes from: health_score, which the paper says is a
FlyRank composite built from "Impressions (30 pts) + position (30 pts)
+ CTR (20 pts) + scroll depth (20 pts)." Does the validation design
carry the claim? No — this is a leakage problem the paper itself
flags: "health score already includes some visibility inputs...
importance is descriptive rather than causal." Predicting a target
partly built from the same features you're using to predict it is
close to circular. High feature importance here isn't a discovery,
it's confirming that the label formula uses those inputs. This is
exactly the "product decision as feature/label" trap from the lane
guide, section 4.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

Below, my Week-5 Random Forest is re-run comparing a plain random
split (naive, "before") against the client-grouped split (honest,
"after") to show why the split choice matters.

In [1]:
!git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
%cd flyRank-internship


Cloning into 'flyRank-internship'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 144 (delta 58), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.86 MiB | 10.66 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/flyRank-internship


In [2]:
# If fresh session, run this first:
# !git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
# %cd flyRank-internship

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")

y = (df["trend_direction"] == "down").astype(int)
num_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                 "avg_position", "ctr", "word_count"]
X = df[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# BEFORE: naive random split (no client grouping)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
rf_naive = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_naive.fit(X_train_r, y_train_r)
probs_naive = rf_naive.predict_proba(X_test_r)[:, 1]

# AFTER: client-grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
probs_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

for k in (20, 50):
    p_naive = precision_at_k(probs_naive, y_test_r.values, k)
    p_grouped = precision_at_k(probs_grouped, y_test_g.values, k)
    print(f"Precision@{k}: naive random split = {p_naive:.3f}  vs  client-grouped split = {p_grouped:.3f}")

Precision@20: naive random split = 1.000  vs  client-grouped split = 0.600
Precision@50: naive random split = 0.940  vs  client-grouped split = 0.680


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

Same leakage hunt as Week 3, applied to the final feature set used
in modeling: content_age_days, days_since_last_update,
impressions_90d, avg_position, ctr, word_count.

In [3]:
leakage_checklist = {
    "Are any features calculated after the decision point?":
        "No - all features come from the same 90-day window used to define the label, not after it.",
    "Does the feature window overlap the target window?":
        "Yes, partially - this is a known limitation. trend_direction (the label) is computed "
        "from the same current window as the features, not a genuinely future outcome. "
        "A stronger design would use prior-90-days features to predict a next-30-days label.",
    "Did any product decision output (health_score, priority_score, action_type) slip in as a feature?":
        "No - none of these fields are in the starter dataset or the feature list used.",
    "Does a derived field secretly encode the target?":
        "No - trend_direction is the target itself, not hidden inside another feature.",
    "Are duplicate/related rows split across train and test unfairly?":
        "Addressed - client-grouped split keeps all of a client's pages on one side only.",
}

for q, a in leakage_checklist.items():
    print(f"Q: {q}\nA: {a}\n")

Q: Are any features calculated after the decision point?
A: No - all features come from the same 90-day window used to define the label, not after it.

Q: Does the feature window overlap the target window?
A: Yes, partially - this is a known limitation. trend_direction (the label) is computed from the same current window as the features, not a genuinely future outcome. A stronger design would use prior-90-days features to predict a next-30-days label.

Q: Did any product decision output (health_score, priority_score, action_type) slip in as a feature?
A: No - none of these fields are in the starter dataset or the feature list used.

Q: Does a derived field secretly encode the target?
A: No - trend_direction is the target itself, not hidden inside another feature.

Q: Are duplicate/related rows split across train and test unfairly?
A: Addressed - client-grouped split keeps all of a client's pages on one side only.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

Bold version (avoid this): "Our model predicts which pages will
decline."

Careful rewrite: "Under a client-grouped holdout split, our Random
Forest model ranked pages by an observed proxy for decline
(trend_direction) with a measured Precision@50 of [insert your
number] on this starter sample - meaning of the top 50 flagged pages,
that share currently show declining behavior in the same window used
to build the label. This is decision-support for prioritizing manual
review, not a forecast of future outcomes, and it has not been tested
against a genuinely future-window label."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.